# RoadAnomaly dataset exploration

**Role.** Scientific presentation and exploration client for the read-only dataset audit in
`003-dataset-understanding`. It reads the canonical JSON and source frames; it does not calculate
authoritative audit statistics, fit a model, choose a metric, create a split, tune a threshold or
select a model.

**Canonical evidence.**

- machine-readable evidence: `docs/validation/roadanomaly-eda.json`
- derived report: `docs/validation/roadanomaly-eda.md`
- audit implementation: `dl/data/audit.py`
- schema: `specs/003-dataset-understanding/contracts/audit-summary.schema.json`

**Evidence classes used in this notebook.** `computed_local`, `documented_local`,
`upstream_unverified` and `unresolved`. Interpretations and hypotheses are marked separately; they
are not new audit evidence.

**Research questions.**

1. What does the benchmark contain, and how consistent are its image and annotation structures?
2. Where does annotated anomaly area and object scale concentrate, and which label definitions
   affect interpretation?
3. Which provenance, licensing and method limits constrain claims and future experiments?


## 1. Research objective and dataset description

### 1.1 Why dataset understanding comes first

The RoadAnomaly benchmark is evaluation-only. Nothing is fitted, selected, tuned, scored or
promoted on its frames. A descriptive audit is the laboratory notebook before an experiment: it
records what is in the sample, what the labels mean, which checks passed and what cannot be checked
locally. It cannot establish predictive performance, generalisation, calibration or model ranking.

The audit therefore stays in CRISP-DM phase 2 (data understanding). It does not decide a class map,
a metric, an operating threshold, a split or a normal-road source. The open scientific items
P0-1 through P0-4, G10 and the ADR-0010 threshold contradiction remain open.

### 1.2 What the dataset checkout contains

The local checkout contains a frame list, WebP road images, one label directory per frame with a
semantic mask and an optional instance mask, an attribution file and a dataset readme. The audit
verifies the local inventory, the image and label structure, the instance/semantic relationship and
the documented provenance. The canonical JSON is a single evidence source for inventory,
fingerprint, integrity, coverage, categories, object scale, instance mapping, ROI bounds, duplicate
checks, credits, provenance, documented-claim comparisons, limitations and claims.

The Markdown report is rendered from the validated JSON bytes and records their SHA-256. This
notebook adds explanation and visual inspection; it does not add an authoritative number.

### 1.3 Scope and reading guide

Sections 2-6 show observed evidence. Section 7 separates limitations, interpretations and
hypotheses. Sections 8-9 discuss implications and conclusions without selecting a model,
architecture, split, threshold or source. Section 10 records the reproducible workflow.

The first analysis cell locates the repository from an environment hint or from the current
working directory. If the canonical pair is missing, invalid or stale, execution stops with an
actionable message rather than continuing with misleading findings.


## 2. Audit status and dataset integrity

### 2.1 Question and prerequisite

**Question.** Is the canonical audit present, schema-valid and consistent with the derived report?
**Method.** Read the JSON bytes, validate them with the project's audit validator, compute the JSON
SHA-256 and compare it with the report's source digest. Re-render the report from the validated
JSON and require byte equality. This is an integrity check, not a second computation of audit
statistics.


In [ ]:
from __future__ import annotations

import contextlib
import hashlib
import importlib
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from PIL import Image


# Resolve the checkout without assuming that the kernel starts in the repository root.
def _find_repo_root() -> Path:
    starts: list[Path] = []
    env_root = os.environ.get("ROADANOMALY_ROOT")
    if env_root:
        starts.append(Path(env_root).expanduser().resolve())
    starts.append(Path.cwd().resolve())
    with contextlib.suppress(NameError):
        starts.append(Path(__file__).resolve().parent)

    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "pyproject.toml").is_file() and (
                candidate / "data" / "frame_list.json"
            ).is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Start the notebook from the checkout or set "
        "ROADANOMALY_ROOT to the directory containing pyproject.toml and data/frame_list.json."
    )


ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

audit = importlib.import_module("dl.data.audit")

DATA_ROOT = ROOT / "data"
JSON_PATH = ROOT / "docs" / "validation" / "roadanomaly-eda.json"
REPORT_PATH = ROOT / "docs" / "validation" / "roadanomaly-eda.md"
SCHEMA_PATH = (
    ROOT / "specs" / "003-dataset-understanding" / "contracts" / "audit-summary.schema.json"
)

for required in (JSON_PATH, REPORT_PATH, SCHEMA_PATH):
    if not required.is_file():
        raise FileNotFoundError(
            f"Missing canonical audit prerequisite: {required}. "
            "Run `make data-audit`, inspect the output, then restart the kernel."
        )

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print(f"Repository root: {ROOT}")
print(f"Canonical JSON: {JSON_PATH.relative_to(ROOT)}")

In [ ]:
RAW_JSON = JSON_PATH.read_bytes()
SUMMARY = json.loads(RAW_JSON)

try:
    audit.validate_summary(SUMMARY)
except audit.AuditContractError as exc:
    raise RuntimeError(
        "The canonical audit JSON is not schema-valid. Run `make data-audit-check`, repair the "
        "evidence through the audit workflow, and restart the kernel."
    ) from exc

JSON_DIGEST = hashlib.sha256(RAW_JSON).hexdigest()
REPORT_TEXT = REPORT_PATH.read_text(encoding="utf-8")
digest_match = re.search(r"Source JSON SHA-256: `([0-9a-f]{64})`", REPORT_TEXT)
if digest_match is None:
    raise RuntimeError(
        "The canonical report has no source JSON SHA-256. The pair is incomplete or stale; "
        "run `make data-audit-check` and do not edit the report by hand."
    )

REPORT_DIGEST = digest_match.group(1)
if REPORT_DIGEST != JSON_DIGEST:
    raise RuntimeError(
        "The report source digest does not match the canonical JSON. This is a stale or mixed "
        "pair; run `make data-audit-check` and regenerate through `make data-audit`."
    )

try:
    RERENDERED_REPORT = audit.render_report(SUMMARY, JSON_DIGEST)
except audit.AuditError as exc:
    raise RuntimeError("The report could not be re-rendered from the validated JSON.") from exc
if RERENDERED_REPORT != REPORT_TEXT:
    raise RuntimeError(
        "The committed report is not byte-identical to a fresh render of the canonical JSON. "
        "The pair is stale or mixed; run `make data-audit-check`."
    )

print("Canonical JSON: schema-valid")
print(f"Source JSON SHA-256: {JSON_DIGEST}")
print("Report source digest: match")
print("Report re-render: byte-identical")

In [ ]:
def _md(value: Any) -> str:
    if value is None:
        return "n/a"
    text = str(value)
    return text.replace("|", r"\|").replace("\n", " ")


def show_table(headers: list[str], rows: list[tuple[Any, ...]]) -> None:
    lines = [
        "| " + " | ".join(headers) + " |",
        "|" + "|".join("---" for _ in headers) + "|",
    ]
    for row in rows:
        lines.append("| " + " | ".join(_md(value) for value in row) + " |")
    display(Markdown("\n".join(lines)))


inventory = SUMMARY["inventory"]
integrity = inventory["annotation_integrity"]
inventory_consistent = (
    inventory["frame_count"]
    == inventory["frame_list_count"]
    == inventory["label_dir_count"]
    == len(inventory["frames"])
)
annotation_ok = all(value for key, value in integrity.items() if isinstance(value, bool))
fingerprint_ok = bool(
    SUMMARY["fingerprint"]["matches"] and SUMMARY["fingerprint"]["status"] == "match"
)

show_table(
    ["Check", "Status", "Detail"],
    [
        ("Canonical JSON", "pass", "schema-validated in the previous cell"),
        ("Report source digest", "pass", f"`{JSON_DIGEST}`"),
        ("Report re-render", "pass", "byte-identical from validated JSON"),
        (
            "Fingerprint",
            "pass" if fingerprint_ok else "fail",
            f"`{SUMMARY['fingerprint']['status']}`",
        ),
        (
            "Inventory consistency",
            "pass" if inventory_consistent else "fail",
            f"{inventory['frame_count']} frames / {len(inventory['frames'])} records",
        ),
        (
            "Annotation integrity",
            "pass" if annotation_ok else "fail",
            "all recorded boolean checks true",
        ),
    ],
)

environment = SUMMARY["environment"]
show_table(
    ["Environment field", "Value", "Evidence"],
    [
        ("Source commit", f"`{environment['commit']}`", "computed_local"),
        ("Python", environment["python"], "computed_local"),
        ("NumPy", environment["numpy"], "computed_local"),
        ("Pillow", environment["pillow"], "computed_local"),
        ("jsonschema", environment["jsonschema"], "computed_local"),
        ("Audit version", environment["audit_version"], "computed_local"),
        ("Canonical command", f"`{environment['command']}`", "computed_local"),
        ("Schema", f"`{SCHEMA_PATH.relative_to(ROOT)}`", "documented_local"),
    ],
)

In [ ]:
fingerprint = SUMMARY["fingerprint"]
show_table(
    ["Fingerprint field", "Value", "Evidence"],
    [
        ("Historical value", f"`{fingerprint['historical']}`", "computed_local"),
        (
            "Historical counts",
            f"{fingerprint['historical_file_count']:,} files, "
            f"{fingerprint['historical_total_bytes']:,} bytes",
            "computed_local",
        ),
        ("Computed value", f"`{fingerprint['computed']}`", "computed_local"),
        (
            "Current counts",
            f"{fingerprint['current_file_count']:,} files, "
            f"{fingerprint['current_total_bytes']:,} bytes",
            "computed_local",
        ),
        ("Match status", fingerprint["status"], "computed_local"),
        ("Recipe version", fingerprint["recipe_version"], "documented_local"),
        ("Provenance", fingerprint["provenance"], "upstream_unverified"),
        ("Recipe", f"`{fingerprint['recipe']}`", "documented_local"),
    ],
)

show_table(
    ["Inventory field", "Value", "Evidence"],
    [
        ("Frame-list entries", inventory["frame_list_count"], "computed_local"),
        ("Frames on disk", inventory["frame_count"], "computed_local"),
        ("Label directories", inventory["label_dir_count"], "computed_local"),
        ("Instance-annotated frames", inventory["instance_frame_count"], "computed_local"),
        ("Semantic-only frames", inventory["semantic_only_frame_count"], "computed_local"),
    ],
)

show_table(
    ["Annotation-integrity field", "Value", "Evidence"],
    [
        ("Semantic values", integrity["semantic_values"], "computed_local"),
        ("Colour mapping", integrity["colour_mapping"], "computed_local"),
        ("Colour mapping valid", integrity["colour_mapping_valid"], "computed_local"),
        ("Instance class ID", integrity["instance_class_id"], "computed_local"),
        ("Contiguous instance IDs", integrity["instance_ids_contiguous"], "computed_local"),
        ("Label files complete", integrity["label_files_complete"], "computed_local"),
        ("Frames decodable", integrity["frame_decodable"], "computed_local"),
    ],
)

### 2.2 Reading the status table

A **pass** means the audit field or pair-level check is present and consistent. A **failed**
fingerprint, annotation invariant or check-mode comparison is a hard audit failure: the audit
exits non-zero and does not produce a misleading canonical pair. **Skipped** is not a state used
here; this notebook has no skip branch. **Unavailable** input stops the notebook before a table is
shown, with the missing path and the command to run.

`environment.commit` is recorded provenance, not a revalidated Git-ancestry claim. Check mode pins
it to the committed value so the evidence remains checkable after `HEAD` advances. The historical
fingerprint recipe is a local reconstruction; matching it does not prove the original Stage 2
implementation byte layout.

The integrity checks above confirm that the pair is internally consistent. They do not confirm
visual registration of frames, semantic masks and instance masks; that is explicitly unresolved in
the audit limitations.


## 3. Annotation coverage and category distributions

### 3.1 Metric definitions and units

**Question.** How much of each frame is annotated as anomaly, and how does that area vary across
the benchmark? **Unit of analysis.** A pixel for coverage, a frame for the per-frame distribution
and a filename prefix for the category counts.

The audit records the numerator, denominator and serialization rule for each coverage quantity.
The pooled value is total anomaly pixels divided by total frame pixels. A per-frame value is that
frame's anomaly pixels divided by its pixel count. The median is taken over per-frame fractions,
and the sub-1% list uses the per-frame percent predicate. All values shown below are read from the
canonical JSON, not recomputed in the notebook.

**Missing versus negative.** A semantic value of zero is a valid normal (negative) annotation, not
missing data. Missing frame or label files are hard failures and are not represented as zero.
Semantic-only frames have semantic labels but no instance IDs; they are not unannotated frames.


In [ ]:
coverage = SUMMARY["coverage"]
show_table(
    ["Coverage quantity", "Value", "Unit / denominator", "Evidence"],
    [
        (
            "Total anomaly pixels",
            f"{coverage['total_anomaly_pixels']:,}",
            "pixels over all frames",
            "computed_local",
        ),
        (
            "Total frame pixels",
            f"{coverage['total_pixels']:,}",
            "pixels over all frames",
            "computed_local",
        ),
        ("Pooled fraction", f"{coverage['pooled_fraction']:.8f}", "fraction", "computed_local"),
        ("Pooled percent", f"{coverage['pooled_percent']:.6f}", "percent", "computed_local"),
        (
            "Minimum per-frame percent",
            f"{coverage['minimum_percent']:.6f}",
            "percent, one frame",
            "computed_local",
        ),
        (
            "Median per-frame percent",
            f"{coverage['median_percent']:.6f}",
            "percent, median of per-frame fractions",
            "computed_local",
        ),
        (
            "Maximum per-frame percent",
            f"{coverage['maximum_percent']:.6f}",
            "percent, one frame",
            "computed_local",
        ),
        (
            "Frames below 1 percent",
            f"{len(coverage['below_one_percent'])}",
            "frames, per-frame percent < 1.0",
            "computed_local",
        ),
    ],
)

show_table(
    ["Definition key", "Audit definition text"],
    sorted(coverage["definitions"].items()),
)

In [ ]:
coverage_rows = coverage["per_frame"]
if not coverage_rows:
    raise RuntimeError("The canonical coverage record has no per-frame rows.")

x = np.arange(1, len(coverage_rows) + 1)
percent = np.asarray([row["percent"] for row in coverage_rows], dtype=float)
below_names = set(coverage["below_one_percent"])
below_mask = np.asarray([row["frame_name"] in below_names for row in coverage_rows])

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(x, percent, color="#4C78A8", linewidth=1.1, label="frame coverage")
ax.scatter(x[~below_mask], percent[~below_mask], color="#4C78A8", s=20)
ax.scatter(
    x[below_mask],
    percent[below_mask],
    color="#D55E00",
    marker="D",
    s=48,
    label="below 1 percent",
)
ax.axhline(1.0, color="#D55E00", linestyle="--", linewidth=1.0, label="1 percent reference")
ax.axhline(
    coverage["median_percent"],
    color="#009E73",
    linestyle=":",
    linewidth=1.4,
    label=f"median {coverage['median_percent']:.6f} percent",
)
ax.axhline(
    coverage["pooled_percent"],
    color="#CC79A7",
    linestyle="-.",
    linewidth=1.2,
    label=f"pooled {coverage['pooled_percent']:.6f} percent",
)
ax.set_xlim(0, len(coverage_rows) + 1)
ax.set_ylim(0, max(5.0, float(percent.max()) * 1.08))
ax.set_xlabel("Frame index in frame_list.json order")
ax.set_ylabel("Annotated anomaly area (percent of frame)")
ax.set_title(f"Per-frame annotated anomaly coverage (n={len(coverage_rows)} frames)")
ax.legend(loc="upper right", frameon=False, fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)

show_table(
    ["Frame below 1 percent", "Percent", "Evidence"],
    [
        (row["frame_name"], f"{row['percent']:.6f}", "computed_local")
        for row in coverage_rows
        if row["frame_name"] in below_names
    ],
)

### 3.2 What the coverage view does and does not show

The per-frame series varies substantially, and a small set of frames sits below the one-percent
reference. The pooled value and the per-frame distribution answer different questions: pooling
gives total annotated area, while the distribution shows that an average is not a typical frame.
A relative error on a low-coverage frame can look large even when the absolute pixel error is small;
that is a measurement consideration, not a model result.

The audit does not infer model performance, failure rates or generalisation from these values. The
benchmark is evaluation-only, and the descriptive distribution cannot rank models or choose a
threshold.


In [ ]:
categories = SUMMARY["categories"]
counts = categories["counts"]
ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
labels = [name for name, _ in ordered]
values = [count for _, count in ordered]

fig, ax = plt.subplots(figsize=(7.2, 4.2))
bars = ax.barh(labels, values, color="#4C78A8")
ax.invert_yaxis()
ax.set_xlabel("Frames")
ax.set_ylabel("Filename-prefix object category")
ax.set_title(f"Object categories, not scene IDs (n={categories['total_frames']} frames)")
ax.set_xlim(0, max(values) * 1.15 if values else 1)
for bar, value in zip(bars, values, strict=True):
    ax.text(
        bar.get_width() + max(values) * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{value}",
        va="center",
    )
fig.tight_layout()
plt.show()
plt.close(fig)

show_table(
    ["Object category", "Frames", "Evidence"],
    [(name, count, "computed_local") for name, count in ordered],
)
display(
    Markdown(
        f"Audit classification: `{categories['classification']}`. Counts use the filename prefix "
        "before the first digit. They are object categories, not scene identities, split units or "
        "source groups."
    )
)

### 3.3 Category interpretation

The category chart is a descriptive count of filename prefixes. It does not show difficulty,
model error, scene diversity or source diversity. The audit explicitly labels these prefixes as
object categories, while source groups are attribution-derived groups. Neither is a substitute
for a scene manifest.

The imbalance is a property of the local benchmark sample. It is relevant when planning metric
aggregation and error analysis, but it does not decide a class weighting, loss, threshold or model.


## 4. Object counts and scale distributions

### 4.1 Question and available evidence

**Question.** How many instances are annotated, and how small are their regions under the audit's
named definitions? **Unit.** One annotated instance for the counts and pixels for the size
summaries. The denominator is the number of instances in the instance-annotated frames.

The canonical JSON reports aggregate small-object counts and medians. It does not expose a
per-instance size vector. This notebook therefore does not reconstruct a histogram from
`index.json`: doing so would create a second, unvalidated evidence source and duplicate audit
logic. The medians and counts below are read directly from the audit.


In [ ]:
scale = SUMMARY["object_scale"]
denominator = scale["denominator"]
scale_rows = [
    ("ROI area < 1024 px^2", scale["roi_area_below_1024"]),
    ("Both ROI dimensions < 32 px", scale["both_dimensions_below_32"]),
    ("Either ROI dimension < 32 px", scale["either_dimension_below_32"]),
]

fig, ax = plt.subplots(figsize=(7.6, 3.8))
labels = [name for name, _ in scale_rows]
values = [value for _, value in scale_rows]
bars = ax.barh(labels, values, color=["#4C78A8", "#009E73", "#CC79A7"])
ax.invert_yaxis()
ax.set_xlim(0, denominator)
ax.set_xlabel(f"Instances (denominator = {denominator})")
ax.set_ylabel("Small-object definition")
ax.set_title("Small-object counts under three definitions")
for bar, value in zip(bars, values, strict=True):
    ax.text(
        bar.get_width() + denominator * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value}/{denominator}",
        va="center",
    )
fig.tight_layout()
plt.show()
plt.close(fig)

show_table(
    ["Scale summary", "Value", "Unit", "Definition"],
    [
        (
            "Median ROI equivalent square",
            f"{scale['median_roi_equivalent_square_px']:.1f}",
            "px",
            scale["definitions"]["roi_equivalent_square"],
        ),
        (
            "Median ROI width",
            f"{scale['median_roi_width_px']:.1f}",
            "px",
            scale["definitions"]["roi_bounding_box"],
        ),
        (
            "Median ROI height",
            f"{scale['median_roi_height_px']:.1f}",
            "px",
            scale["definitions"]["roi_bounding_box"],
        ),
        (
            "Median mask equivalent square",
            f"{scale['median_mask_equivalent_square_px']:.1f}",
            "px",
            scale["definitions"]["mask_equivalent_square"],
        ),
        (
            "Instances in scale denominator",
            f"{denominator}",
            "instances",
            "one row per instance in instance-annotated frames",
        ),
    ],
)
show_table(
    ["Scale definition key", "Audit definition text"],
    sorted(scale["definitions"].items()),
)

### 4.2 Reading the scale evidence

The three small-object definitions do not select the same instances. A count under "ROI area <
1024 px^2" is not interchangeable with a count under "both dimensions < 32 px" or "either
dimension < 32 px". The median ROI bounding-box dimensions and the median mask-equivalent square
also describe different objects: the ROI is source annotation metadata, while the mask summary
uses the stored instance pixels.

The canonical JSON does not provide a per-object distribution, so no tail shape, outlier count or
quantile beyond the reported medians is claimed. The figures are descriptive and cannot be used to
infer segmentation quality, model sensitivity or generalisation. A future analysis that needs an
object-size distribution must add a separately reviewed evidence path.


## 5. Region of Interest (ROI) analysis

### 5.1 ROI definitions and assumptions

**Question.** Are the stored ROI rectangles consistent with the image frame and with the stored
instance pixels? **Unit.** One annotated instance in an instance-annotated frame.

A ROI is a rectangle in `index.json`; it is source annotation metadata, not a predicted box. The
audit uses half-open extents, so a ROI ending exactly at the image width or height is inside. A
pixel box is non-tight when its minimum or maximum coordinate falls outside the ROI's half-open
range. Centroid containment uses the midpoint of the instance pixel bounding box. The audit does
not clip, repair or renumber any annotation.

The order mapping compares mask value `k` with `instances[k-1].id`. That mapping is not universally
stable in this checkout, so the non-tight interpretation is conditional on the documented mapping.


In [ ]:
roi = SUMMARY["roi"]
denominator = roi["instances_checked"]
show_table(
    ["ROI finding", "Count", "Denominator", "Evidence"],
    [
        ("Out-of-bounds ROIs", roi["out_of_bounds"], denominator, "computed_local"),
        ("Non-tight instance boxes", roi["non_tight"], denominator, "computed_local"),
        ("Centroids inside the frame", roi["centroids_inside"], denominator, "computed_local"),
    ],
)
show_table(
    ["ROI definition key", "Audit definition text"],
    sorted(roi["definitions"].items()),
)

fig, ax = plt.subplots(figsize=(7.6, 3.8))
labels = ["Out of bounds", "Non-tight", "Centroid inside"]
values = [roi["out_of_bounds"], roi["non_tight"], roi["centroids_inside"]]
bars = ax.bar(labels, values, color=["#D55E00", "#E69F00", "#009E73"])
ax.set_ylim(0, denominator)
ax.set_ylabel(f"Instances (n={denominator})")
ax.set_title("ROI findings are separate properties, not a partition")
for bar, value in zip(bars, values, strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + denominator * 0.012,
        f"{value}/{denominator}",
        ha="center",
    )
fig.tight_layout()
plt.show()
plt.close(fig)

out_rows = sorted(
    roi["out_of_bounds_instances"], key=lambda item: (item["frame_name"], item["mask_value"])
)
non_tight_rows = sorted(
    roi["non_tight_instances"], key=lambda item: (item["frame_name"], item["mask_value"])
)
show_table(
    ["Finding", "Frame", "Mask value", "Evidence"],
    [("out_of_bounds", row["frame_name"], row["mask_value"], "computed_local") for row in out_rows]
    + [
        ("non_tight", row["frame_name"], row["mask_value"], "computed_local")
        for row in non_tight_rows
    ],
)

### 5.2 What the ROI findings mean

The counts and complete lists show that ROI bounds and instance-pixel tightness are not uniformly
consistent. An out-of-bounds ROI is not silently clipped by the audit, and a non-tight box is not
silently repaired. Because the order mapping disagrees in some frames, an object-level ROI claim
must state which mask-to-entry mapping it uses.

Plausible explanations include editor or resizing conventions, different coordinate origins, or a
mapping mismatch. Those are hypotheses, not established causes. The audit's finding is the
inconsistency itself. No visual registration judgment is made here.


## 6. Visual inspection of images and annotations

### 6.1 Sampling and display contract

**Question.** Do representative source images and stored annotations look structurally coherent
at the pixel level? The sample is deterministic: the first frame, the minimum-coverage frame, the
maximum-coverage frame, the frame closest to the median coverage, the first semantic-only frame
and the first frame with an order-mapping disagreement. Frames are de-duplicated in that order.

Each triptych shows the unmodified source frame, the original semantic color annotation and the
stored instance mask. Instance colors are a derived display of stored integer values; no label is
edited. These panels are not overlays and do not verify registration. Attribution comes from the
canonical credit records. If a file is unavailable, the panel reports it instead of substituting
data.


In [ ]:
inventory = SUMMARY["inventory"]
frames = inventory["frames"]
coverage_rows = coverage["per_frame"]
if not frames or not coverage_rows:
    raise RuntimeError("The canonical inventory or coverage rows are empty.")

frame_by_name = {frame["frame_name"]: frame for frame in frames}
coverage_by_name = {row["frame_name"]: row for row in coverage_rows}
credit_by_name = {entry["frame_name"]: entry for entry in SUMMARY["credits"]["entries"]}
mapping_counts = Counter(
    mismatch["frame_name"] for mismatch in SUMMARY["instance_mapping"]["mismatches"]
)
roi = SUMMARY["roi"]

selected: dict[str, str] = {}


def add_selection(name: str, reason: str) -> None:
    selected.setdefault(name, reason)


add_selection(frames[0]["frame_name"], "first frame in frame_list.json")
minimum_name = next(
    row["frame_name"] for row in coverage_rows if row["percent"] == coverage["minimum_percent"]
)
add_selection(minimum_name, "minimum per-frame coverage")
maximum_name = next(
    row["frame_name"] for row in coverage_rows if row["percent"] == coverage["maximum_percent"]
)
add_selection(maximum_name, "maximum per-frame coverage")
median_name = sorted(
    coverage_rows,
    key=lambda row: (abs(row["percent"] - coverage["median_percent"]), row["frame_name"]),
)[0]["frame_name"]
add_selection(median_name, "closest per-frame coverage to the audit median")
semantic_name = next(
    frame["frame_name"] for frame in frames if frame["annotation_kind"] == "semantic_only"
)
add_selection(semantic_name, "first semantic-only frame")
if SUMMARY["instance_mapping"]["disagreeing_frames"]:
    add_selection(
        sorted(SUMMARY["instance_mapping"]["disagreeing_frames"])[0],
        "first frame with an order-mapping disagreement",
    )

show_table(
    ["Selected frame", "Reason", "Annotation", "Coverage percent", "Mapping mismatches"],
    [
        (
            name,
            reason,
            frame_by_name[name]["annotation_kind"],
            f"{coverage_by_name[name]['percent']:.6f}",
            mapping_counts.get(name, 0),
        )
        for name, reason in selected.items()
    ],
)

In [ ]:
def safe_data_path(relative: str) -> Path:
    path = (DATA_ROOT / relative).resolve()
    if not path.is_relative_to(DATA_ROOT.resolve()):
        raise ValueError(f"Canonical path escapes the data root: {relative!r}")
    return path


for name, reason in selected.items():
    record = frame_by_name[name]
    coverage_row = coverage_by_name.get(name)
    credit = credit_by_name.get(name)
    label_dir = safe_data_path(record["label_dir"])
    frame_path = safe_data_path(record["frame_path"])

    details = [
        f"### `{name}`",
        f"*Selection:* {reason}; *annotation kind:* `{record['annotation_kind']}`.",
    ]
    if coverage_row is not None:
        details.append(
            f"*Audit coverage:* {coverage_row['percent']:.6f} percent (`computed_local`)."
        )
    if mapping_counts.get(name, 0):
        details.append(
            f"*Audit order-mapping mismatches in this frame:* {mapping_counts[name]} "
            "(`computed_local`)."
        )
    out_count = sum(1 for item in roi["out_of_bounds_instances"] if item["frame_name"] == name)
    tight_count = sum(1 for item in roi["non_tight_instances"] if item["frame_name"] == name)
    if out_count or tight_count:
        details.append(
            f"*ROI findings:* out-of-bounds {out_count}; non-tight {tight_count} "
            "(`computed_local`)."
        )
    if credit is None:
        details.append(
            "*Attribution:* no credit record (`unresolved`). Do not republish this image as if "
            "attribution were complete."
        )
    else:
        tokens = ", ".join(credit["licence_tokens"]) if credit["licence_tokens"] else "none"
        source_url = credit["source_url"] or "none"
        details.append(f"*Credit tokens:* {tokens}; *source URL:* {source_url} (`computed_local`).")
        details.append(f"*Credit block:* {credit['credit_text'].replace(chr(10), ' / ')}")
    details.append(
        "*Visual note:* source frame and stored masks are shown separately; this is not an "
        "overlay and does not assert registration."
    )
    display(Markdown("  \n".join(details)))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    try:
        with Image.open(frame_path) as image:
            axes[0].imshow(np.asarray(image.convert("RGB")))
        axes[0].set_title("Source frame (unmodified)")

        semantic_color = label_dir / "labels_semantic_color.png"
        semantic_gray = label_dir / "labels_semantic.png"
        if semantic_color.is_file():
            with Image.open(semantic_color) as image:
                axes[1].imshow(np.asarray(image.convert("RGB")))
        elif semantic_gray.is_file():
            with Image.open(semantic_gray) as image:
                axes[1].imshow(np.asarray(image.convert("L")), cmap="gray")
        else:
            axes[1].text(
                0.5,
                0.5,
                "Semantic annotation unavailable",
                ha="center",
                va="center",
            )
        axes[1].set_title("Original semantic annotation")

        if record["annotation_kind"] == "instance":
            instance_path = label_dir / "labels_instance.png"
            if instance_path.is_file():
                with Image.open(instance_path) as image:
                    instance = np.asarray(image.convert("L"))
                instance_plot = np.where(instance == 0, np.nan, instance)
                instance_cmap = plt.get_cmap("tab20").copy()
                instance_cmap.set_bad("black")
                axes[2].imshow(
                    instance_plot,
                    cmap=instance_cmap,
                    vmin=1,
                    vmax=max(1, int(instance.max())),
                    interpolation="nearest",
                )
                axes[2].set_title("Original instance IDs (derived colors)")
            else:
                axes[2].text(
                    0.5,
                    0.5,
                    "Instance annotation unavailable",
                    ha="center",
                    va="center",
                )
        else:
            axes[2].text(
                0.5,
                0.5,
                "No instance annotation\n(semantic-only frame)",
                ha="center",
                va="center",
            )
            axes[2].set_title("Instance annotation")
    except Exception as exc:
        for axis in axes:
            axis.axis("off")
        axes[0].text(
            0.5,
            0.5,
            f"Display unavailable:\n{type(exc).__name__}: {exc}",
            ha="center",
            va="center",
            wrap=True,
        )
        axes[0].set_title("Unavailable")
    else:
        for axis in axes:
            axis.axis("off")

    fig.suptitle(name, fontsize=10)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

### 6.2 Limits of visual inspection

These panels are a small deterministic sample, not a visual review of all frames. They show the
stored annotations without correction and without overlaying them on the source image. A panel can
look plausible while a coordinate or registration error remains; conversely, a mask can look
coarse without being invalid under the label contract. Visual registration, a review ledger and
overlay figures remain post-MVP extensions and are not claimed as verified here.


## 7. Limitations, potential biases, and open questions

### 7.1 What the audit explicitly does not establish

The audit records method limits instead of turning them into passing claims. The table below is
read from the canonical `limitations` and `provenance` fields. Missing per-image licence tokens
remain `unresolved`, not "unlicensed". Source groups are attribution-derived, not scene identities.


In [ ]:
show_table(
    ["Limitation", "Statement", "Evidence", "Source"],
    [
        (
            limitation["limitation_id"],
            limitation["statement"],
            limitation["evidence_class"],
            limitation["source"] or "n/a",
        )
        for limitation in SUMMARY["limitations"]
    ],
)
show_table(
    ["Provenance field", "Value", "Evidence"],
    [
        ("Citation", SUMMARY["provenance"]["citation"], "documented_local"),
        ("Release identifier", SUMMARY["provenance"]["release_identifier"], "upstream_unverified"),
        ("Upstream status", SUMMARY["provenance"]["upstream_status"], "upstream_unverified"),
        ("Scene identity", SUMMARY["provenance"]["scene_identity"], "unresolved"),
        (
            "Filename prefix is scene ID",
            SUMMARY["provenance"]["filename_prefix_is_scene_id"],
            "computed_local",
        ),
        (
            "Local attribution is redistribution proof",
            SUMMARY["provenance"]["local_attribution_is_redistribution_proof"],
            "upstream_unverified",
        ),
        (
            "Normal-road source selected",
            SUMMARY["provenance"]["normal_road_source_selected"],
            "computed_local",
        ),
    ],
)
show_table(
    ["Open item", "Status", "Evidence"],
    [
        (key, value, "unresolved")
        for key, value in sorted(SUMMARY["provenance"]["p0_status"].items())
    ]
    + [("G10", SUMMARY["provenance"]["g10_status"], "unresolved")]
    + [
        (
            item["item_id"],
            item["status"],
            item["evidence_class"],
        )
        for item in SUMMARY["provenance"]["unresolved_items"]
    ],
)

In [ ]:
credits = SUMMARY["credits"]
show_table(
    ["Credit and licensing field", "Value", "Evidence"],
    [
        ("Frames with credit entries", credits["coverage"], "computed_local"),
        ("Missing credit entries", len(credits["missing_entries"]), "computed_local"),
        (
            "Frames lacking explicit licence tokens",
            len(credits["missing_licence_tokens"]),
            "unresolved",
        ),
        ("Frames lacking source URLs", len(credits["missing_source_urls"]), "unresolved"),
        ("Annotation licence claim", credits["annotation_licence_claim"], "documented_local"),
        (
            "Annotation licence sources",
            ", ".join(credits["annotation_licence_sources"]),
            "documented_local",
        ),
        (
            "Annotation terms URL",
            credits["annotation_licence_terms_url"],
            "documented_local",
        ),
        ("Code licence", credits["code_licence"], "documented_local"),
        ("Code licence source", credits["code_licence_source"], "documented_local"),
        ("Code licence scope", credits["code_licence_scope"], "documented_local"),
        ("Legal conclusion", credits["legal_conclusion"], "unresolved"),
    ],
)
show_table(
    ["Source group", "Frames", "Source URL", "Evidence"],
    [
        (
            group["group_id"],
            ", ".join(group["frames"]),
            group["source_url"] or "none (unresolved)",
            "computed_local",
        )
        for group in credits["source_groups"]
    ],
)
show_table(
    ["Frame lacking explicit licence token", "Evidence"],
    [(name, "unresolved") for name in credits["missing_licence_tokens"]],
)
show_table(
    ["Frame lacking source URL", "Evidence"],
    [(name, "unresolved") for name in credits["missing_source_urls"]],
)
display(
    Markdown(
        "Missing tokens remain unresolved, not unlicensed. Local attribution is not a legal "
        "conclusion or redistribution proof. The annotation and code licence statements are "
        "`documented_local`, not independently verified legal advice."
    )
)

### 7.2 Documented claims that must not be silently reconciled

The audit compares local computed values with statements in the local documentation. Both sides
are shown with their evidence class and treatment. This notebook does not rewrite the source
documents and does not choose one interpretation as the only truth.


In [ ]:
show_table(
    ["Claim", "Documented text", "Computed value", "Treatment"],
    [
        (
            claim["claim_id"],
            claim["documented_text"],
            claim["computed_value"],
            claim["treatment"],
        )
        for claim in SUMMARY["documented_claims"]
    ],
)
display(
    Markdown(
        "Documented text is `documented_local`; computed values are `computed_local`; an unresolved "
        "licence or provenance item stays `unresolved` or `upstream_unverified`."
    )
)

### 7.3 Observations, interpretations and hypotheses

| Layer | Statement |
|---|---|
| Observation | The canonical inventory and annotation-integrity checks are internally consistent, and the instance/semantic invariant is exact for the audited instance frames. |
| Interpretation | Structural consistency does not remove the ambiguity in instance-ID ordering or the provenance gaps; those are separate label and evidence questions. |
| Hypothesis | The order-mapping disagreement may reflect editor/export ordering or rescaled annotation history. This is not verified locally. |
| Implication | Any object-level metric or ROI interpretation must state and validate its mask-to-entry mapping before comparing models. |
| Observation | ROI bounds and pixel-box tightness are not uniformly consistent across instances. |
| Interpretation | The findings may come from coordinate conventions or resizing rather than label error; the audit reports them without repair. |
| Hypothesis | ROI rectangles and instance pixel boxes may follow different storage conventions. This requires source-editor or upstream metadata to test. |
| Implication | Do not clip, repair or silently reinterpret ROIs in a future evaluation; define the convention first. |
| Observation | Credit coverage is complete at the block level, but explicit per-image licence tokens are uneven. |
| Interpretation | Local attribution is useful provenance but not a legal redistribution conclusion, and missing tokens are unresolved rather than proof of no licence. |
| Hypothesis | Some missing tokens may be recoverable from upstream pages; this is outside the offline audit. |
| Implication | Figures and redistributed derivatives need the credit block and an explicit unresolved-licence flag. |
| Observation | The reported exact, decoded-pixel and dHash duplicate lists are empty under the stated methods. |
| Interpretation | This supports "no duplicate found by these methods", not "no near-duplicate exists". |
| Hypothesis | Near-duplicates could exist below the dHash distance or outside the checked transforms. |
| Implication | A leakage-free claim needs a separately defined and validated duplicate protocol. |
| Observation | Visual registration, scene identity and normal-road readiness are not assessed. |
| Interpretation | The local files can be structurally validated without proving that annotations align visually or that a training boundary exists. |
| Hypothesis | Human review would reveal registration issues not visible in aggregate checks. |
| Implication | Keep those as extensions/open questions; do not treat the audit as a training-data sign-off. |

The ADR-0010 threshold contradiction, P0-1 through P0-4 and G10 remain open. Nothing in this table
resolves them.


## 8. Implications for future Deep Learning experiments

### 8.1 A useful analogy, with its limit

A pre-flight checklist records the state of the aircraft, instruments and weather; it does not
prove that a flight will succeed. The dataset audit is similar: it records inventory, label
structure, coverage, object scale, ROI findings, duplicates and provenance so that an experiment
does not start from an unexamined sample. The analogy stops there. A checklist is not a performance
measurement, and neither is this audit.

Precisely: these descriptive statistics constrain what can be measured and where labels are
ambiguous. They do not estimate accuracy, generalisation, calibration, robustness or model ranking.

### 8.2 Evidence-based considerations

The table below converts observed audit evidence into considerations, not decisions. It does not
choose an architecture, loss, metric, threshold, split or normal-road source.

The order-mapping issue is like two street maps that number the same building differently: the
city is the same, but a route instruction is ambiguous until the numbering convention is fixed.
Here, mask value `k` and the `k`-th index entry do not always agree, so object-level claims need a
documented mapping before comparison.


In [ ]:
credits = SUMMARY["credits"]
mapping = SUMMARY["instance_mapping"]
roi = SUMMARY["roi"]
inventory = SUMMARY["inventory"]
coverage = SUMMARY["coverage"]
scale = SUMMARY["object_scale"]
normal_road_open = not SUMMARY["provenance"]["normal_road_source_selected"]

show_table(
    ["Topic", "Audit evidence", "Consideration for future experiments"],
    [
        (
            "Pixel imbalance",
            f"pooled {coverage['pooled_percent']:.6f} percent; per-frame "
            f"{coverage['minimum_percent']:.6f}-{coverage['maximum_percent']:.6f} percent; "
            f"{len(coverage['below_one_percent'])} frames below 1 percent",
            "Report both pooled and per-frame values; define the denominator and uncertainty. "
            "Do not infer accuracy from coverage or choose a loss here.",
        ),
        (
            "Semantic-only frames",
            f"{inventory['semantic_only_frame_count']} frames have semantic labels but no "
            "instance annotation",
            "Frame-level semantic evaluation can include them; instance-level evaluation cannot. "
            "Do not fabricate instance labels or silently drop the frames.",
        ),
        (
            "Instance ordering",
            f"{mapping['frames_agreeing']}/{mapping['frames_checked']} frames agree under the "
            f"k-th-entry mapping; {mapping['frames_disagreeing']} disagree",
            "Object-level metrics, ROI analysis and per-instance visualization need a documented "
            "mapping rule before comparison. The audit does not renumber masks.",
        ),
        (
            "ROI bounds",
            f"{roi['out_of_bounds']} out-of-bounds ROIs; {roi['non_tight']} non-tight boxes; "
            f"{roi['centroids_inside']}/{roi['instances_checked']} centroids inside",
            "Define half-open handling and a localization policy. Do not clip or repair "
            "annotations silently; report the convention with any object-level result.",
        ),
        (
            "Object scale",
            f"ROI area < 1024: {scale['roi_area_below_1024']}/{scale['denominator']}; both "
            f"dimensions < 32: {scale['both_dimensions_below_32']}; either: "
            f"{scale['either_dimension_below_32']}; median ROI equivalent square "
            f"{scale['median_roi_equivalent_square_px']:.1f} px",
            "Choose and document one size definition if a future analysis needs it. The three "
            "definitions are not interchangeable and do not predict error.",
        ),
        (
            "Duplicates",
            "exact, decoded-pixel and dHash lists are empty under the stated methods",
            "Treat absence as method-limited. A leakage claim needs a stronger, reviewed protocol; "
            "do not assume the benchmark is duplicate-free.",
        ),
        (
            "Credits and licensing",
            f"{len(credits['missing_licence_tokens'])} frames lack explicit licence tokens; "
            f"{len(credits['source_groups'])} attribution groups cover "
            f"{sum(group['frame_count'] for group in credits['source_groups'])} frames; "
            f"{len(credits['missing_source_urls'])} lack a source URL",
            "Carry attribution with figures and keep missing tokens unresolved. Local attribution "
            "is not a legal conclusion or redistribution proof.",
        ),
        (
            "Evaluation-only boundary",
            f"normal-road source selected: {SUMMARY['provenance']['normal_road_source_selected']}; "
            f"G10: {SUMMARY['provenance']['g10_status']}",
            "Do not train, select, tune or promote on the benchmark. Model selection needs a "
            "pinned normal-road validation boundary, which is not present locally.",
        ),
    ],
)
display(
    Markdown(
        "These are considerations for a future approved experiment, not architecture, split, "
        "threshold or source decisions. No model quality or generalisation is inferred."
    )
)

## 9. Evidence-based conclusions and next steps

### 9.1 Separate established findings from open questions

The conclusion cell reads the canonical values and marks their evidence class. It does not add a
new statistic. The benchmark remains evaluation-only, and no model performance, generalisation or
ranking claim is made.


In [ ]:
coverage = SUMMARY["coverage"]
inventory = SUMMARY["inventory"]
scale = SUMMARY["object_scale"]
mapping = SUMMARY["instance_mapping"]
roi = SUMMARY["roi"]
credits = SUMMARY["credits"]
invariant = SUMMARY["instance_semantic"]
provenance = SUMMARY["provenance"]

display(
    Markdown(
        f"""
### Established findings (`computed_local` unless marked)

- The canonical JSON is schema-valid, and the report is a byte-identical render of its validated
  bytes with the source digest `{JSON_DIGEST}`.
- Inventory: {inventory["frame_count"]} frames, {inventory["label_dir_count"]} label directories,
  {inventory["instance_frame_count"]} instance-annotated frames and
  {inventory["semantic_only_frame_count"]} semantic-only frames.
- Instance/semantic invariant: {invariant["frames_exact_match"]}/{invariant["frames_checked"]}
  exact, {invariant["extra_instance_pixels"]} extra pixels and
  {invariant["missing_instance_pixels"]} missing pixels; {invariant["semantic_only_frames_excluded"]}
  semantic-only frames excluded.
- Coverage: {coverage["pooled_percent"]:.6f} percent pooled,
  {coverage["minimum_percent"]:.6f}-{coverage["maximum_percent"]:.6f} percent per frame,
  {coverage["median_percent"]:.6f} percent median, and
  {len(coverage["below_one_percent"])} frames below 1 percent.
- Object categories: the audit reports
  {", ".join(f"{name} {count}" for name, count in sorted(SUMMARY["categories"]["counts"].items()))}
  as filename-prefix object categories, not scenes.
- Object scale: ROI area &lt; 1024 px^2 {scale["roi_area_below_1024"]}/{scale["denominator"]},
  both dimensions &lt; 32 px {scale["both_dimensions_below_32"]}/{scale["denominator"]}, either
  dimension &lt; 32 px {scale["either_dimension_below_32"]}/{scale["denominator"]}; medians
  {scale["median_roi_equivalent_square_px"]:.1f} px ROI equivalent square,
  {scale["median_roi_width_px"]:.1f} x {scale["median_roi_height_px"]:.1f} px ROI bounding box and
  {scale["median_mask_equivalent_square_px"]:.1f} px mask equivalent square.
- Instance mapping: {mapping["frames_agreeing"]}/{mapping["frames_checked"]} frames agree and
  {mapping["frames_disagreeing"]} disagree under the k-th-entry comparison.
- ROI: {roi["out_of_bounds"]}/{roi["instances_checked"]} out-of-bounds,
  {roi["non_tight"]}/{roi["instances_checked"]} non-tight and
  {roi["centroids_inside"]}/{roi["instances_checked"]} centroids inside.
- Duplicates: the audit reported {len(SUMMARY["duplicates"]["exact_pairs"])} exact,
  {len(SUMMARY["duplicates"]["decoded_pairs"])} decoded-pixel and
  {len(SUMMARY["duplicates"]["dhash_pairs"])} dHash pairs under the stated methods. This is
  method-limited, not proof of absence.
- Credits: {credits["coverage"]} entries; {len(credits["missing_licence_tokens"])} frames lack
  explicit licence tokens; {len(credits["source_groups"])} attribution groups cover
  {sum(group["frame_count"] for group in credits["source_groups"])} frames; and
  {len(credits["missing_source_urls"])} frames lack a source URL.
- Visual registration, scene identity and normal-road readiness are unresolved/not assessed.

### Hypotheses and open questions (`unresolved`)

- Does the instance-order disagreement reflect editor history, a different numbering convention or
  a data defect? The local evidence cannot distinguish these explanations.
- Do ROI rectangles and instance pixel boxes follow different coordinate or resizing conventions?
- Could near-duplicates exist outside the exact, decoded-pixel and dHash checks?
- Are filename prefixes usable as scenes? No local scene metadata supports that claim.
- Can missing per-image licence tokens be recovered upstream? This audit does not establish it.

### Next steps, not decisions

- Keep the benchmark evaluation-only; do not train, tune, select or promote on it.
- Resolve P0-1 through P0-4, G10 and the ADR-0010 threshold contradiction through the project's
  approved process.
- If visual registration is required, authorize the separate visual-review extension; this
  notebook does not replace it.
- If normal-road readiness is required, authorize the separate readiness extension; no source is
  selected here.

No model quality, generalisation or benchmark ranking is claimed.
"""
    )
)

## 10. Reproducibility and workflow

### 10.1 Canonical audit workflow

The audit is generated and verified from the repository root. The canonical JSON is authoritative;
the Markdown report is derived from validated JSON bytes. The JSON does not contain a report hash,
and the report contains the JSON source digest.


In [ ]:
environment = SUMMARY["environment"]
display(
    Markdown(
        f"""
### Commands and provenance

- Generate: `make data-audit`
- Verify without writing: `make data-audit-check`
- Canonical JSON: `docs/validation/roadanomaly-eda.json`
- Derived report: `docs/validation/roadanomaly-eda.md`
- Audit implementation: `dl/data/audit.py`
- Audit version: `{environment["audit_version"]}`
- Canonical command recorded in JSON: `{environment["command"]}`
- Source commit provenance: `{environment["commit"]}` (not a Git-ancestry revalidation)
- Source JSON SHA-256: `{JSON_DIGEST}`

### Fresh-kernel execution

1. Install the repository environment (the project's `make install` target, or the declared
   requirements files). Jupyter frontend/kernel packages are environment tooling and are not pinned
   by the audit.
2. Start Jupyter from the repository root, or set `ROADANOMALY_ROOT` to the checkout. The first
   cells resolve paths from that root rather than from relative working-directory assumptions.
3. Restart the kernel and run all cells. The notebook reads canonical JSON and source frames; it
   writes no files, saves no figures and creates no experiment record.
4. If the canonical pair is missing, invalid or stale, the notebook stops. Run
   `make data-audit-check`; if regeneration is required, use `make data-audit` and inspect both
   outputs before rerunning. Do not hand-edit either file.

### Scope checks

- [x] No model fitting, tuning, inference, selection or promotion.
- [x] No split, class map, split hash, manifest or experiment record.
- [x] No source acquisition, ranking or normal-road selection.
- [x] No overlay, review ledger, aggregate figure export or readiness report.
- [x] P0-1 through P0-4, G10 and the ADR-0010 threshold contradiction stay open.
- [x] The notebook is a presentation client; `docs/validation/roadanomaly-eda.json` is canonical.
"""
    )
)